In [ ]:
# === Parent: configure ===

# Name (or full workspace path) of the child notebook to call.
# In Fabric, pass just the notebook display name if it's in the same workspace.
child_notebook = "02-child-call-azure-function"

# Per-call timeout in seconds (0 = infinite)
child_timeout_s = 300

# Parameters the child will receive. The child should declare matching names
# in a cell tagged 'parameters'.
child_params = {
    "function_url":  "https://my-fn-app.azurewebsites.net/api/HelloWorld",
    "function_key":  "REPLACE_WITH_FUNCTION_KEY"
}

print(f"Will call child: {child_notebook}")
print(f"With params keys: {list(child_params.keys())}")

# Parent notebook -> child notebook -> Azure Function

This notebook orchestrates a 3-tier chain:

```
this notebook
   |  notebookutils.notebook.run(...)
   v
02-child-call-azure-function (child notebook)
   |  requests.post(...)
   v
Azure Function (HTTP-triggered)
```

The child is reusable: this parent passes parameters and reads back a JSON result via `notebookutils.notebook.exit()`.

In [ ]:
# === Parent: invoke child notebook ===
# notebookutils.notebook.run is the Fabric replacement for mssparkutils.notebook.run.
# It returns whatever string the child passes to notebookutils.notebook.exit().

try:
    from notebookutils import notebook       # Fabric runtime 1.2+
except Exception:
    from mssparkutils import notebook        # older alias

#p1 = json.dumps(child_params)

raw_result = notebook.run(
    child_notebook,
    child_timeout_s,
    child_params
)

print(f"Child returned (raw, len={len(raw_result)}):")
print(raw_result[:500])

In [ ]:
# === Parent: parse child's JSON result ===
import json

try:
    result = json.loads(raw_result)
except json.JSONDecodeError:
    print("Child did NOT return JSON. Raw value above.")
    raise

print(f"Status      : {result.get('status')}")
print(f"HTTP code   : {result.get('http_status')}")
print(f"Duration ms : {result.get('duration_ms')}")
print()
print("Response body from function:")
print(json.dumps(result.get('response'), indent=2))

# Fail this notebook if the function call failed downstream.
assert result.get('status') == 'ok', f"Function call failed: {result.get('error')}"